# 🧪 A.R.C. — Ablation Testing Notebook

**Purpose:** Test each optimization individually against the reference problems to find the optimal configuration before final submission.

## Test Matrix

| # | Variable | A (baseline) | B (proposed) | Expected |
|---|----------|-------------|-------------|----------|
| 1 | Prompts | Verbose (NB1) | Simple V40 | +1-2 pts |
| 2 | Entropy | Simple mean | 5-component weighted | +1-2 pts |
| 3 | Selection | Entropy only | GenSelect hybrid | +1-2 pts |
| 4 | top_p | 0.8 | None | 0 pts |
| 5 | Z3 import | Without | With | 0 pts (no regression) |

**Rule from Notebook 2:** Test ONE variable at a time. V131 combined 6 changes = -6 regression.

---
## Setup (same for all tests)

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import os, sys, subprocess

def set_env(input_archive, temp_dir):
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', f'{temp_dir}/wheels',
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

import warnings; warnings.simplefilter('ignore')
import gc, re, math, time, queue, threading, contextlib, json
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd
import polars as pl
from openai import OpenAI
from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding,
    SystemContent, ReasoningEffort, ToolNamespaceConfig,
    Author, Message, Role, TextContent, Conversation
)
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

---
## 🔧 Test Control Panel

**Change ONE variable at a time**, run, observe score, then reset.

Toggle the flags below to enable/disable each optimization:

In [ ]:
# ========================================================
# TEST FLAGS — toggle ONE at a time to measure impact
# ========================================================

USE_SIMPLE_PROMPTS  = True   # True=V40 simple, False=NB1 verbose
USE_WEIGHTED_ENTROPY = True  # True=5-component, False=simple mean
USE_GENSELECT       = True   # True=hybrid selection, False=entropy only
USE_TOP_P           = False  # True=top_p=0.8, False=no top_p
USE_Z3_IMPORT       = True   # True=import z3, False=skip

print('Current test configuration:')
print(f'  Simple prompts:    {USE_SIMPLE_PROMPTS}')
print(f'  Weighted entropy:  {USE_WEIGHTED_ENTROPY}')
print(f'  GenSelect:         {USE_GENSELECT}')
print(f'  top_p:             {"0.8" if USE_TOP_P else "None"}')
print(f'  Z3 import:         {USE_Z3_IMPORT}')

---
## Prompt Variants

In [ ]:
# V40-style simple prompts (proven: 42/50)
SIMPLE_SYSTEM = (
    'You are a world-class IMO competitor. '
    'The final answer must be 0-99999. '
    'Place answer inside \\boxed{}.'
)
SIMPLE_TOOL = (
    'Use this tool to execute Python code. '
    'Use print() to output results.'
)
SIMPLE_PREF = 'You have access to math, numpy, sympy.'

# NB1 verbose prompts (proven: 44/50, but see NB2 evidence)
VERBOSE_SYSTEM = (
    'You are an elite mathematical problem solver with expertise at the International '
    'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
    'rigorous mathematical reasoning.\n\n'
    '# Problem-Solving Approach:\n'
    '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
    'Identify what is given, what needs to be found, and any constraints.\n'
    '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
    'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
    '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
    '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
    '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
    'alternative methods. Ensure logical consistency throughout.\n\n'
    '# Mathematical Reasoning Principles:\n'
    '- Break complex problems into smaller, manageable sub-problems\n'
    '- Look for patterns, symmetries, and special cases that provide insight\n'
    '- Use concrete examples to build intuition before generalizing\n'
    '- Consider extreme cases and boundary conditions\n'
    '- If stuck, try working backwards from the desired result\n'
    '- Be willing to restart with a different approach if needed\n\n'
    '# Verification Requirements:\n'
    '- Cross-check arithmetic and algebraic manipulations\n'
    '- Verify that your solution satisfies all problem constraints\n'
    '- Test your answer with simple cases or special values when possible\n'
    '- Ensure dimensional consistency and reasonableness of the result\n\n'
    '# Output Format:\n'
    'The final answer must be a non-negative integer between 0 and 99999.\n'
    'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
    'Think step-by-step and show your complete reasoning process. Quality of reasoning '
    'is as important as the final answer.'
)
VERBOSE_TOOL = (
    'Use this tool to execute Python code for:\n'
    '- Complex calculations that would be error-prone by hand\n'
    '- Numerical verification of analytical results\n'
    '- Generating examples or testing conjectures\n'
    '- Visualizing problem structure when helpful\n'
    '- Brute-force verification for small cases\n\n'
    'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
    'Always use print() to display results. Write clear, well-commented code.\n\n'
    'Remember: Code should support your mathematical reasoning, not replace it. '
    'Explain what you\'re computing and why before running code.'
)
VERBOSE_PREF = (
    'You have access to `math`, `numpy`, and `sympy` for:\n\n'
    '# Symbolic Computation (sympy):\n'
    '- Algebraic manipulation and simplification\n'
    '- Solving equations and systems of equations\n'
    '- Symbolic differentiation and integration\n'
    '- Number theory functions (primes, divisors, modular arithmetic)\n'
    '- Polynomial operations and factorization\n'
    '- Working with mathematical expressions symbolically\n\n'
    '# Numerical Computation (numpy):\n'
    '- Array operations and linear algebra\n'
    '- Efficient numerical calculations for large datasets\n'
    '- Matrix operations and eigenvalue problems\n'
    '- Statistical computations\n\n'
    '# Mathematical Functions (math):\n'
    '- Standard mathematical functions (trig, log, exp)\n'
    '- Constants like pi and e\n'
    '- Basic operations for single values\n\n'
    'Best Practices:\n'
    '- Use sympy for exact symbolic answers when possible\n'
    '- Use numpy for numerical verification and large-scale computation\n'
    '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
    '- Document your computational strategy clearly\n'
    '- Validate computational results against known cases or theoretical bounds'
)

---
## Entropy Variants

In [ ]:
def compute_simple_mean_entropy(logprobs_buffer: list) -> float:
    """NB1 original: simple mean of per-token entropies."""
    if not logprobs_buffer:
        return float('inf')
    total_entropy = 0.0
    token_count = 0
    for top_logprobs_dict in logprobs_buffer:
        if not isinstance(top_logprobs_dict, dict) or not top_logprobs_dict:
            continue
        token_entropy = 0.0
        for token_str, log_prob in top_logprobs_dict.items():
            prob = math.exp(log_prob)
            if prob > 0:
                token_entropy -= prob * math.log2(prob)
        total_entropy += token_entropy
        token_count += 1
    return total_entropy / token_count if token_count > 0 else float('inf')


def compute_weighted_entropy(logprobs_buffer: list) -> float:
    """NB2 5-component weighted entropy (proven +5-6 pts)."""
    if not logprobs_buffer:
        return float('inf')
    entropies = []
    for top_logprobs_dict in logprobs_buffer:
        if not isinstance(top_logprobs_dict, dict) or not top_logprobs_dict:
            continue
        token_entropy = 0.0
        for _, log_prob in top_logprobs_dict.items():
            prob = math.exp(log_prob)
            if prob > 0:
                token_entropy -= prob * math.log2(prob)
        entropies.append(token_entropy)
    if not entropies:
        return float('inf')
    n = len(entropies)
    mean_ent = sum(entropies) / n
    variance = sum((e - mean_ent) ** 2 for e in entropies) / n
    std_dev = math.sqrt(variance)
    decay = 0.995
    w = [decay ** (n - 1 - i) for i in range(n)]
    tw = sum(w)
    pos_weighted = sum(wi * e for wi, e in zip(w, entropies)) / tw
    high_ratio = sum(1 for e in entropies if e > 2.0) / n
    max_streak = cur = 0
    for e in entropies:
        if e < 1.0:
            cur += 1
            max_streak = max(max_streak, cur)
        else:
            cur = 0
    streak_bonus = -0.1 * (max_streak / n)
    return 0.3*mean_ent + 0.4*pos_weighted + 0.2*std_dev + 0.3*high_ratio*3.0 + streak_bonus


def compute_entropy(logprobs_buffer: list) -> float:
    """Dispatch based on test flag."""
    if USE_WEIGHTED_ENTROPY:
        return compute_weighted_entropy(logprobs_buffer)
    else:
        return compute_simple_mean_entropy(logprobs_buffer)

---
## Config (reads test flags)

In [ ]:
class CFG:
    # Prompts — selected by test flag
    system_prompt = SIMPLE_SYSTEM if USE_SIMPLE_PROMPTS else VERBOSE_SYSTEM
    tool_prompt   = SIMPLE_TOOL   if USE_SIMPLE_PROMPTS else VERBOSE_TOOL
    preference_prompt = SIMPLE_PREF if USE_SIMPLE_PROMPTS else VERBOSE_PREF

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'
    gpu_memory_utilization = 0.96
    context_tokens = 65536
    temperature = 1.0
    min_p = 0.02

    high_problem_timeout = 900
    base_problem_timeout = 300
    notebook_limit = 17400
    server_timeout = 180
    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3
    stream_interval = 200
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    genselect_enabled = USE_GENSELECT
    genselect_temperature = 0.1
    genselect_max_tokens = 512
    genselect_trace_limit = 4000
    genselect_top_k = 3

set_seed(CFG.seed)
print(f'Prompts: {"SIMPLE" if USE_SIMPLE_PROMPTS else "VERBOSE"}')
print(f'Entropy: {"5-COMPONENT" if USE_WEIGHTED_ENTROPY else "SIMPLE MEAN"}')
print(f'GenSelect: {"ON" if USE_GENSELECT else "OFF"}')

---
## Core Components
*(Same as notebook.ipynb — Template, Sandbox, Tool, Selector, Solver)*

**Note:** The only differences from `notebook.ipynb` are:
1. Entropy function dispatches based on `USE_WEIGHTED_ENTROPY` flag
2. GenSelect on/off via `USE_GENSELECT` flag
3. Prompts switch via `USE_SIMPLE_PROMPTS` flag

Copy the Template, Sandbox, Tool, GenSelector, select_answer, and Solver cells from `notebook.ipynb` here — they reference `compute_entropy()` and `CFG` which we've already set up above to respect the test flags.

For brevity, see `notebook.ipynb` for the full implementations.

In [ ]:
# === Chat Template ===
class AIMO3Template:
    def get_system_content(self, system_prompt, tool_config):
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )
    def apply_chat_template(self, system_prompt, user_prompt, tool_config):
        system_content = self.get_system_content(system_prompt, tool_config)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

# === Sandbox ===
class AIMO3Sandbox:
    _port_lock = threading.Lock()
    _next_port = 50000
    PREAMBLE = (
        "import math\nimport numpy\nimport sympy\nimport itertools\n"
        "import collections\nimport mpmath\nmpmath.mp.dps = 64\n"
        "try:\n    from z3 import *\nexcept ImportError:\n    pass\n"
    ) if USE_Z3_IMPORT else (
        "import math\nimport numpy\nimport sympy\nimport itertools\n"
        "import collections\nimport mpmath\nmpmath.mp.dps = 64\n"
    )
    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports
    def __init__(self, timeout):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
        env["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "0"
        env["JUPYTER_PLATFORM_DIRS"] = "1"
        env["PYTHONWARNINGS"] = "ignore"
        env["MPLBACKEND"] = "Agg"
        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]
        self._km.start_kernel(env=env, extra_arguments=["--Application.log_level=CRITICAL"])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute(self.PREAMBLE)
    def _format_error(self, traceback):
        clean = []
        for frame in traceback:
            f = re.sub(r"\x1b\[[0-9;]*m", "", frame)
            if "File \"" in f and "ipython-input" not in f: continue
            clean.append(f)
        return "".join(clean)
    def execute(self, code, timeout=None):
        client = self._client
        t = timeout or self._default_timeout
        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout_parts, stderr_parts = [], []
        start = time.time()
        while True:
            if time.time() - start > t:
                self._km.interrupt_kernel()
                return f"[ERROR] Timed out after {t}s"
            try: msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty: continue
            if msg.get("parent_header",{}).get("msg_id") != msg_id: continue
            mt = msg.get("msg_type"); c = msg.get("content",{})
            if mt=="stream":
                (stdout_parts if c.get("name")=="stdout" else stderr_parts).append(c.get("text",""))
            elif mt=="error": stderr_parts.append(self._format_error(c.get("traceback",[])))
            elif mt in {"execute_result","display_data"}:
                t2 = c.get("data",{}).get("text/plain")
                if t2: stdout_parts.append(t2 if t2.endswith("\n") else f"{t2}\n")
            elif mt=="status" and c.get("execution_state")=="idle": break
        so = "".join(stdout_parts); se = "".join(stderr_parts)
        if se: return f"{so.rstrip()}\n{se}" if so else se
        return so if so.strip() else "[WARN] No output. Use print()."
    def reset(self): self.execute("%reset -f\n" + self.PREAMBLE)
    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()
    def __del__(self): self.close()

# === Tool Bridge ===
class AIMO3Tool:
    def __init__(self, local_jupyter_timeout, tool_prompt, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()
    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)
    def _ensure_last_print(self, code):
        lines = code.strip().split("\n")
        if not lines: return code
        last = lines[-1].strip()
        if "print" in last or "import" in last or not last or last.startswith("#"): return code
        lines[-1] = "print(" + last + ")"
        return "\n".join(lines)
    @property
    def instruction(self): return self._tool_prompt
    @property
    def tool_config(self): return ToolNamespaceConfig(name="python", description=self.instruction, tools=[])
    def _make_response(self, output, channel=None):
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name="python")
        message = Message(author=author, content=[content]).with_recipient("assistant")
        if channel: message = message.with_channel(channel)
        return message
    def process_sync_plus(self, message):
        self._ensure_session()
        raw = message.content[0].text
        final = self._ensure_last_print(raw)
        with self._execution_lock:
            try: output = self._jupyter_session.execute(final)
            except TimeoutError as e: output = f"[ERROR] {e}"
        return [self._make_response(output, channel=message.channel)]

# === GenSelector ===
class GenSelector:
    def __init__(self, client, cfg, encoding):
        self.client = client; self.cfg = cfg; self.encoding = encoding
    def select(self, problem, candidates):
        groups = defaultdict(list)
        for c in candidates:
            if c.get("Answer") is not None: groups[c["Answer"]].append(c)
        if len(groups) <= 1: return list(groups.keys())[0] if groups else None
        ranked = sorted(groups.items(), key=lambda x: sum(1.0/max(c.get("Entropy",float("inf")),1e-9) for c in x[1]), reverse=True)[:self.cfg.genselect_top_k]
        prompt = self._build_prompt(problem, ranked)
        try:
            resp = self.client.completions.create(model=self.cfg.served_model_name, prompt=prompt, temperature=self.cfg.genselect_temperature, max_tokens=self.cfg.genselect_max_tokens, extra_body={"min_p":self.cfg.min_p})
            sel = scan_for_answer(resp.choices[0].text)
            if sel is not None and sel in groups: return sel
        except: pass
        return None
    def _build_prompt(self, problem, ranked_groups):
        p = [f"Problem: {problem}\n\nMultiple solutions produced different answers. Select the most rigorous.\n\n"]
        for i,(ans,traces) in enumerate(ranked_groups):
            best = min(traces, key=lambda t: t.get("Entropy",float("inf")))
            p.append(f"=== Solution {i+1} (Answer: {ans}, Votes: {len(traces)}) ===\n{best.get(chr(84)+chr(114)+chr(97)+chr(99)+chr(101),chr(32))[:self.cfg.genselect_trace_limit]}\n\n")
        p.append("Which is most rigorous? Reply with \\boxed{answer}.")
        return "".join(p)

def select_answer(detailed_results, problem="", gen_selector=None):
    answer_weights = defaultdict(float); answer_votes = defaultdict(int)
    for r in detailed_results:
        a = r.get("Answer"); e = r.get("Entropy", float("inf"))
        if a is not None: answer_weights[a] += 1.0/max(e,1e-9); answer_votes[a] += 1
    if not answer_weights: return 0
    top = max(answer_votes, key=answer_votes.get)
    if answer_votes[top] >= 4: print(f"  [CONSENSUS] {top}"); return top
    if gen_selector and len(answer_votes)>=2 and USE_GENSELECT:
        try:
            gs = gen_selector.select(problem, detailed_results)
            if gs is not None: print(f"  [GENSELECT] {gs}"); return gs
        except: pass
    scored = sorted([{"a":a,"v":answer_votes[a],"s":s} for a,s in answer_weights.items()], key=lambda x:x["s"], reverse=True)
    display(pd.DataFrame(scored)); w = scored[0]["a"]; print(f"  [ENTROPY] {w}"); return w

# === Main Solver ===
class AIMO3Solver:
    def __init__(self, cfg, port=8000):
        self.cfg = cfg; self.port = port
        self.base_url = f"http://0.0.0.0:{port}/v1"; self.api_key = "sk-local"
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server(); self._initialize_kernels()
        self.gen_selector = GenSelector(self.client, self.cfg, self.encoding) if self.cfg.genselect_enabled else None
        self.notebook_start_time = time.time(); self.problems_remaining = 50
    def _preload_model_weights(self):
        print(f"Loading weights from {self.cfg.model_path}...")
        start = time.time(); files = []
        for root,_,fs in os.walk(self.cfg.model_path):
            for fn in fs:
                fp = os.path.join(root,fn)
                if os.path.isfile(fp): files.append(fp)
        def _read(p):
            with open(p,"rb") as f:
                while f.read(1024*1024*1024): pass
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex: list(ex.map(_read, files))
        print(f"Loaded {len(files)} files in {time.time()-start:.2f}s")
    def _start_server(self):
        cmd = [sys.executable,"-m","vllm.entrypoints.openai.api_server","--seed",str(self.cfg.seed),"--model",self.cfg.model_path,"--served-model-name",self.cfg.served_model_name,"--tensor-parallel-size","1","--max-num-seqs",str(self.cfg.batch_size),"--gpu-memory-utilization",str(self.cfg.gpu_memory_utilization),"--host","0.0.0.0","--port",str(self.port),"--dtype",self.cfg.dtype,"--kv-cache-dtype",self.cfg.kv_cache_dtype,"--max-model-len",str(self.cfg.context_tokens),"--stream-interval",str(self.cfg.stream_interval),"--async-scheduling","--disable-log-stats","--enable-prefix-caching"]
        self.log_file = open("vllm_server.log","w")
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
    def _wait_for_server(self):
        print("Waiting for vLLM server...")
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.server_process.poll() is not None:
                self.log_file.flush()
                with open("vllm_server.log") as f: raise RuntimeError(f"Server died:\n{f.read()}")
            try: self.client.models.list(); print(f"Server ready in {time.time()-start:.2f}s"); return
            except: time.sleep(1)
        raise RuntimeError("Server timeout")
    def _initialize_kernels(self):
        print(f"Initializing {self.cfg.workers} kernels...")
        start = time.time(); self.sandbox_pool = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            for f in as_completed([ex.submit(lambda: AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)) for _ in range(self.cfg.workers)]):
                self.sandbox_pool.put(f.result())
        print(f"Kernels ready in {time.time()-start:.2f}s")
    def _process_attempt(self, problem, system_prompt, attempt_index, stop_event, deadline):
        if stop_event.is_set() or time.time()>deadline:
            return {"Attempt":attempt_index+1,"Answer":None,"Python Calls":0,"Python Errors":0,"Response Length":0,"Entropy":float("inf"),"Trace":""}
        sandbox=None; py_calls=0; py_errors=0; total_tok=0; final_ans=None; lp_buf=[]; all_txt=[]
        seed = int(math.pow(self.cfg.seed+attempt_index,2))
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, tool_prompt=self.cfg.tool_prompt, sandbox=sandbox)
            enc = self.encoding
            msgs = self.template.apply_chat_template(system_prompt, problem, tool.tool_config)
            conv = Conversation.from_messages(msgs)
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time()>deadline: break
                pid = enc.render_conversation_for_completion(conv, Role.ASSISTANT)
                mt = self.cfg.context_tokens - len(pid)
                if mt < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(model=self.cfg.served_model_name, temperature=self.cfg.temperature, logprobs=self.cfg.top_logprobs, max_tokens=mt, prompt=pid, seed=seed, stream=True, extra_body={"min_p":self.cfg.min_p, "stop_token_ids":self.stop_token_ids, "return_token_ids":True})
                try:
                    tok_buf=[]; txt_ch=[]
                    for chunk in stream:
                        if stop_event.is_set() or time.time()>deadline: break
                        nt = chunk.choices[0].token_ids; nx = chunk.choices[0].text
                        if nt:
                            tok_buf.extend(nt); total_tok+=len(nt); txt_ch.append(nx); all_txt.append(nx)
                            cl = chunk.choices[0].logprobs
                            if cl and cl.top_logprobs: lp_buf.extend(cl.top_logprobs)
                        if "}" in nx:
                            st = "".join(txt_ch[-self.cfg.search_tokens:])
                            a = scan_for_answer(st)
                            if a is not None: final_ans=a; break
                finally: stream.close()
                if final_ans is not None: break
                if not tok_buf: break
                new_msgs = enc.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs); last = new_msgs[-1]
                if last.channel=="final":
                    final_ans = scan_for_answer(last.content[0].text); break
                if last.recipient=="python":
                    py_calls+=1
                    tr = tool.process_sync_plus(last)
                    rt = tr[0].content[0].text
                    if rt.startswith("[ERROR]") or "Traceback" in rt or "Error:" in rt: py_errors+=1
                    conv.messages.extend(tr)
        except: py_errors+=1
        finally:
            if sandbox: sandbox.reset(); self.sandbox_pool.put(sandbox)
        ent = compute_entropy(lp_buf)
        return {"Attempt":attempt_index+1,"Response Length":total_tok,"Python Calls":py_calls,"Python Errors":py_errors,"Entropy":ent,"Answer":final_ans,"Trace":"".join(all_txt[-200:])}
    def solve_problem(self, problem):
        print(f"\nProblem: {problem}\n")
        user_input = f"{problem} {self.cfg.preference_prompt}"
        elapsed = time.time()-self.notebook_start_time
        tl = self.cfg.notebook_limit-elapsed
        res = max(0,self.problems_remaining-1)*self.cfg.base_problem_timeout
        budget = min(max(tl-res, self.cfg.base_problem_timeout), self.cfg.high_problem_timeout)
        deadline = time.time()+budget
        print(f"Budget: {budget:.0f}s | Remaining: {self.problems_remaining}")
        results=[]; valid=[]; stop=threading.Event(); ex=ThreadPoolExecutor(max_workers=self.cfg.workers)
        try:
            futs = [ex.submit(self._process_attempt,user_input,self.cfg.system_prompt,i,stop,deadline) for i in range(self.cfg.attempts)]
            for f in as_completed(futs):
                try:
                    r = f.result(); results.append(r)
                    if r["Answer"] is not None: valid.append(r["Answer"])
                    c = Counter(valid).most_common(1)
                    if c and c[0][1]>=self.cfg.early_stop:
                        stop.set()
                        for ff in futs: ff.cancel()
                        break
                except Exception as e: print(f"Failed: {e}")
        finally: stop.set(); ex.shutdown(wait=True,cancel_futures=True); self.problems_remaining=max(0,self.problems_remaining-1)
        if results:
            df=pd.DataFrame(results); df["Entropy"]=df["Entropy"].round(3); df["Answer"]=df["Answer"].astype("Int64")
            display(df[["Attempt","Response Length","Python Calls","Python Errors","Entropy","Answer"]])
        if not valid: print("Result: 0"); return 0
        final = select_answer(results, problem, self.gen_selector)
        print(f"Final Answer: {final}"); return final
    def __del__(self):
        if hasattr(self,"server_process"): self.server_process.terminate(); self.server_process.wait()
        if hasattr(self,"log_file"): self.log_file.close()
        if hasattr(self,"sandbox_pool"):
            while not self.sandbox_pool.empty():
                try: self.sandbox_pool.get_nowait().close()
                except: pass


---
## 📊 Test Runner & Results Logger

In [ ]:
class TestLogger:
    """Logs test results for comparison across runs."""
    
    def __init__(self):
        self.results = []
        self.config = {
            'prompts': 'simple' if USE_SIMPLE_PROMPTS else 'verbose',
            'entropy': 'weighted' if USE_WEIGHTED_ENTROPY else 'simple',
            'genselect': USE_GENSELECT,
            'top_p': '0.8' if USE_TOP_P else 'none',
            'z3': USE_Z3_IMPORT
        }
    
    def log(self, problem_id, answer, expected, time_taken, details):
        correct = answer == expected
        self.results.append({
            'id': problem_id,
            'answer': answer,
            'expected': expected,
            'correct': correct,
            'time_s': round(time_taken, 1),
            'details': details
        })
        status = '✅' if correct else '❌'
        print(f'{status} Problem {problem_id}: got {answer}, expected {expected} ({time_taken:.1f}s)')
    
    def summary(self):
        correct = sum(1 for r in self.results if r['correct'])
        total = len(self.results)
        avg_time = sum(r['time_s'] for r in self.results) / total if total else 0
        print(f'\n{"="*60}')
        print(f'CONFIG: {json.dumps(self.config, indent=2)}')
        print(f'SCORE:  {correct}/{total}')
        print(f'AVG TIME: {avg_time:.1f}s per problem')
        print(f'EST TOTAL: {avg_time * 50 / 60:.1f} min for 50 problems')
        print(f'{"="*60}\n')
        
        # Show failures
        failures = [r for r in self.results if not r['correct']]
        if failures:
            print('FAILURES:')
            for f in failures:
                print(f'  Problem {f["id"]}: got {f["answer"]}, expected {f["expected"]}')
        
        return {'config': self.config, 'score': f'{correct}/{total}', 'avg_time': avg_time}

logger = TestLogger()

---
## Run Tests on Reference Problems

In [ ]:
# Load reference problems with known answers
reference = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv')
print(f'Loaded {len(reference)} reference problems')
display(reference.head())

In [ ]:
# Initialize solver (takes ~3-4 min)
solver = AIMO3Solver(CFG)

In [ ]:
# Run on reference problems
for idx, row in reference.iterrows():
    problem_id = row['id']
    question = row['problem']
    expected = row['answer']
    
    start = time.time()
    answer = solver.solve_problem(question)
    elapsed = time.time() - start
    
    logger.log(problem_id, answer, expected, elapsed, {})

# Print summary
result = logger.summary()

---
## 📋 How to Use This Notebook

### Run 1: Baseline (NB1 config)
```python
USE_SIMPLE_PROMPTS  = False  # verbose
USE_WEIGHTED_ENTROPY = False  # simple mean
USE_GENSELECT       = False  # entropy only
```
→ Record score (should match ~44/50 on full set)

### Run 2: Simple prompts only
```python
USE_SIMPLE_PROMPTS  = True   # ← CHANGED
USE_WEIGHTED_ENTROPY = False
USE_GENSELECT       = False
```
→ Record score. Expected: +1-2 pts

### Run 3: + Weighted entropy
```python
USE_SIMPLE_PROMPTS  = True
USE_WEIGHTED_ENTROPY = True   # ← CHANGED
USE_GENSELECT       = False
```
→ Record score. Expected: +1-2 pts more

### Run 4: + GenSelect
```python
USE_SIMPLE_PROMPTS  = True
USE_WEIGHTED_ENTROPY = True
USE_GENSELECT       = True    # ← CHANGED
```
→ Record score. Expected: +1-2 pts more

### Run 5: Final config = best combination from above
→ This becomes the `notebook.ipynb` submission config